In [ ]:
import numpy as np
import pandas as pd


df_cleaned = pd.read_csv("../../data/processed/balanced_dataset_30k.csv")

In [ ]:
df_cleaned.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_cleaned["text"]
y = df_cleaned["label"]

X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_test_val, y_test_val, test_size=0.50, random_state=42, stratify=y_test_val)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=5)
vectorizer.fit(X_train)

In [ ]:
import pickle

with open("../models/tfidf_vectorizer_xgb.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
X_train_vec = vectorizer.transform(X_train)

In [ ]:
X_test_vec = vectorizer.transform(X_test)
X_val_vec = vectorizer.transform(X_val)

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,  # dataset is balanced 50/50
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train_vec, y_train, verbose=False)

In [ ]:
model.score(X_test_vec, y_test)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

In [ ]:
y_val_pred = model.predict(X_val_vec)
print(classification_report(y_val, y_val_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt

# Predictions for confusion matrix and ROC
y_pred = model.predict(X_test_vec)
y_score = model.predict_proba(X_test_vec)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ROC curve
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 most predictive features (XGBoost feature importance)
importances = model.feature_importances_
top_idx = np.argsort(importances)[-20:]
top_features = vectorizer.get_feature_names_out()[top_idx]
top_scores = importances[top_idx]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(20), top_scores[::-1], align="center")
ax.set_yticks(range(20))
ax.set_yticklabels(top_features[::-1])
ax.set_xlabel("Feature Importance")
ax.set_title("Top 20 Predictive Words / Phrases (XGBoost)")
plt.tight_layout()
plt.show()

In [ ]:
with open("../models/xgboost_model.pkl", "wb") as f:
    pickle.dump(model, f)